① Tool Definitions + Messages
개발자가 사용할 수 있는 함수(예: get_weather(location))를 미리 정의
사용자가 질문: “What’s the weather in Paris?”

② Tool Calls
모델이 질문을 보고 “아, 이건 get_weather("paris") 함수를 호출해야겠네”
텍스트가 아니라 함수 호출 요청(JSON) 을 생성

③ Execute Function Code
실제 코드에서 get_weather("paris") 실행
외부 API(OpenWeather 같은) 호출

④ Results (All Prior Messages)
함수 실행 결과가 다시 모델에게 전달
모델은 이제 “파리의 온도 = 14도”라는 사실을 알게 됨

⑤ Final Response
모델이 사용자에게 자연어로 최종 답변 생성
“It’s currently 14°C in Paris.”

- LLM이 API를 직접 실행하는 게 아니라
“어떤 함수를 호출할지 결정”만 하고
실행은 개발자 코드,
결과를 다시 받아 문장 생성
👉 LLM + 외부 시스템 연동 구조

In [21]:
from openai import OpenAI          # OpenAI API를 사용하기 위한 공식 클라이언트 임포트
from dotenv import load_dotenv     # .env 파일에 저장된 환경변수를 불러오는 함수
import os                          # 운영체제 환경변수에 접근하기 위한 표준 라이브러리

load_dotenv()                      # .env 파일을 읽어서 환경변수로 로드
os.environ['OPENAI_API_KEY'] = os.getenv('openai_key')  # .env에 있는 openai_key 값을 OPENAI_API_KEY로 설정
OPENWEATHER_API_KEY = os.getenv('openweather_key')      # .env에 저장된 OpenWeather API 키를 변수로 로드


In [22]:
# OpenWeather API 호출로 부산 날씨 데이터 조회
import requests

city_name = "Busan"
units = 'metric'
url = f"https://api.openweathermap.org/data/2.5/weather?q={city_name}&appid={OPENWEATHER_API_KEY}&units={units}"
response = requests.get(url)
data = response.json()
data

{'coord': {'lon': 129.0403, 'lat': 35.1028},
 'weather': [{'id': 800,
   'main': 'Clear',
   'description': 'clear sky',
   'icon': '01d'}],
 'base': 'stations',
 'main': {'temp': -2.01,
  'feels_like': -6.44,
  'temp_min': -2.01,
  'temp_max': -2.01,
  'pressure': 1030,
  'humidity': 43,
  'sea_level': 1030,
  'grnd_level': 1025},
 'visibility': 10000,
 'wind': {'speed': 3.6, 'deg': 310},
 'clouds': {'all': 0},
 'dt': 1770079356,
 'sys': {'type': 1,
  'id': 8086,
  'country': 'KR',
  'sunrise': 1770070906,
  'sunset': 1770108802},
 'timezone': 32400,
 'id': 1838524,
 'name': 'Busan',
 'cod': 200}

In [23]:
weather_info={}

if response.status_code == 200:
    weather_description = data['weather'][0]['description']
    temp = data['main']['temp']
    temp_feels_like = data['main']['feels_like']
    humidity = data['main']['humidity']
    
    weather_info = {
        'city' : city_name,
        'description': weather_description,
        'temperature':temp,
        'temperature_feels_like':temp_feels_like,
        'humidity': humidity
    }
    
weather_info

{'city': 'Busan',
 'description': 'clear sky',
 'temperature': -2.01,
 'temperature_feels_like': -6.44,
 'humidity': 43}

In [24]:
# 실패시 실패 원인 확인용 디버깅
print("status_code: ", response.status_code)
print("response json :", data)
print("message :", data.get("message", "no message"))

status_code:  200
response json : {'coord': {'lon': 129.0403, 'lat': 35.1028}, 'weather': [{'id': 800, 'main': 'Clear', 'description': 'clear sky', 'icon': '01d'}], 'base': 'stations', 'main': {'temp': -2.01, 'feels_like': -6.44, 'temp_min': -2.01, 'temp_max': -2.01, 'pressure': 1030, 'humidity': 43, 'sea_level': 1030, 'grnd_level': 1025}, 'visibility': 10000, 'wind': {'speed': 3.6, 'deg': 310}, 'clouds': {'all': 0}, 'dt': 1770079356, 'sys': {'type': 1, 'id': 8086, 'country': 'KR', 'sunrise': 1770070906, 'sunset': 1770108802}, 'timezone': 32400, 'id': 1838524, 'name': 'Busan', 'cod': 200}
message : no message


In [25]:
# 날씨 API 호출 함수: doc_string을 활용해 함수 설명
import requests
import json

def get_current_weather(city_name='Seoul', units='metric'):
    """  OpenWeather API를 사용해서 사용자가 지정한 도시의 현재 날씨 정보를 가져오는 함수

    Args:
        city: str 날씨정보를 가져올 도시 이름. 반드시 영문으로 작성하세요.
            변환예시:
                서울 -> Seoul
                충남, 충청남도 -> Chungcheongnam-do
                부산 -> Busan
        units: str 온도단위를 설정하는 문자열
            metric(기본값: 섭씨, 미터)
            imperial(화씨, 야드)
    Return:
            str: json 형식으로 변환된 현재 날씨 정보
    """
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city_name}&appid={OPENWEATHER_API_KEY}&units={units}"
    response = requests.get(url) # API 요청 전송
    data = response.json()   

    if response.status_code == 200:   # 응답 코드 200이면 정상 처리되었다.
        weather_description = data['weather'][0]['description'] # 날씨 설명
        temp = data['main']['temp']                             # 온도
        temp_feels_like = data['main']['feels_like']            # 체감 온도
        humidity = data['main']['humidity']                     # 습도
        

        weather_info = {
            'city' : city_name,
            'description' : weather_description,
            'temperature' : temp,
            'temperature_feels_like' : temp_feels_like,
            'humidity' : humidity    

    }

    else:
        weather_info = {
        'city' : city_name,
        'description' : 'Not Found',
        'temperature' : 'Not Found',
        'temperature_feels_like' : 'Not Found',
        'humidity' : 'Not Found'  
        }

    return json.dumps(weather_info)              # dict를 JSON 문자열로 변환해 반환

In [26]:
# llm이 사용할 함수(tool) 모음
tools_to_execute = {
    'get_current_weather' : get_current_weather,    # 키: LMM이 호출할 tool 이름, 값: 실제 실행될 함수 
}

# LLM 준비

In [27]:
# get_current_weather 함수의 docstring(설명) 출력
print(tools_to_execute['get_current_weather'].__doc__)

  OpenWeather API를 사용해서 사용자가 지정한 도시의 현재 날씨 정보를 가져오는 함수

    Args:
        city: str 날씨정보를 가져올 도시 이름. 반드시 영문으로 작성하세요.
            변환예시:
                서울 -> Seoul
                충남, 충청남도 -> Chungcheongnam-do
                부산 -> Busan
        units: str 온도단위를 설정하는 문자열
            metric(기본값: 섭씨, 미터)
            imperial(화씨, 야드)
    Return:
            str: json 형식으로 변환된 현재 날씨 정보
    


In [28]:
from pprint import pprint                     # 딕셔너리/리스트를 보기 좋게 출력하기 위한 모듈
from openai import OpenAI                    # OpenAI API 클라이언트 클래스

client = OpenAI()                            # OpenAI 클라이언트 생성

# 사용자 질문을 받아 tool 호출 여부를 판단하고 최종 답변까지 생성하는 함수
def run_conversation(user_prompt, model="gpt-4.1-mini"):  # 대화 전체를 실행하는 함수 정의
    messages = [                             # 대화 히스토리를 담을 메시지 리스트
        {
            "role": "system",                # 시스템 역할 메시지
            "content": (
                "당신은 친절한 챗봇입니다. 사용자의 요구를 분석해 직접 대답하거나, "
                "주어진 함수를 이용해 필요한 정보를 먼저 확보한 후 대답하세요."
            ),                               # 모델의 기본 행동 지침
        },
        {
            "role": "user",                  # 사용자 역할 메시지
            "content": user_prompt,          # 실제 사용자 입력 프롬프트
        },
    ]

    # 모델이 호출할 수 있는 함수(tool) 목록 정의
    tools = [
        {
            "type": "function",              # tool 타입은 function
            "function": {
                "name": "get_current_weather",  # 모델이 호출할 함수 이름
                "description": tools_to_execute["get_current_weather"].__doc__,  # 함수 설명(docstring)
                "parameters": {              # 함수 입력 파라미터 스키마
                    "type": "object",
                    "properties": {
                        "city_name": {       # 도시 이름 파라미터
                            "type": "string",
                            "description": (
                                "도시이름(필수값). 반드시 영어로 작성하세요.\n"
                                "변환예시:\n"
                                "  서울 -> Seoul\n"
                                "  충남, 충청남도 -> Chungcheongnam-do\n"
                                "  경남, 경상남도 -> Gyeongsangnam-do\n"
                                "  전남, 전라남도 -> Jeollanam-do\n"
                                "  부산 -> Busan"
                            ),
                        },
                        "units": {           # 온도 단위 파라미터
                            "type": "string",
                            "description": (
                                "온도단위를 설정하는 문자열\n\n"
                                "metric(기본값: 섭씨, 미터)\n"
                                "imperial(화씨, 야드)"
                            ),
                            "enum": ["metric", "imperial"],  # 허용되는 값 제한
                        },
                    },
                    "required": ["city_name"],  # 필수 파라미터 지정
                },
            },
        }
    ]

    response1 = client.chat.completions.create(  # 1차 모델 호출
        model=model,                          # 사용할 모델
        messages=messages,                    # 현재까지의 대화 메시지
        tools=tools                           # 모델이 사용할 수 있는 함수 목록
    )

    response1_message = response1.choices[0].message   # 모델의 첫 번째 응답 메시지
    response1_tool_calls = response1_message.tool_calls  # 모델이 요청한 tool 호출 목록

    if response1_tool_calls:                  # 모델이 함수 호출을 요청한 경우
        messages.append(response1_message)   # assistant 메시지를 대화 히스토리에 추가

        for tool_call in response1_tool_calls:  # 요청된 각 tool 호출에 대해 반복
            function_name = tool_call.function.name   # 호출할 함수 이름
            print(f"[tool] {function_name}을 호출합니다!")  # 로그 출력
            function_to_execute = tools_to_execute[function_name]  # 실제 실행할 함수
            function_args = json.loads(tool_call.function.arguments)  # 함수 인자(JSON) 파싱
            function_response = function_to_execute(**function_args)  # 함수 실행 결과

            messages.append({                # tool 실행 결과를 메시지로 추가
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": function_response
            })
            pprint(messages)                 # 현재 메시지 상태 출력(디버깅용)

        response2 = client.chat.completions.create(  # tool 결과를 포함해 2차 모델 호출
            model=model,
            messages=messages
        )
        return response2.choices[0].message.content  # 최종 자연어 답변 반환

    else:                                    # 함수 호출이 필요 없는 경우
        return response1_message.content     # 1차 응답을 그대로 반환


In [29]:
run_conversation('오늘 서울 날씨는 어때?')

[tool] get_current_weather을 호출합니다!
[{'content': '당신은 친절한 챗봇입니다. 사용자의 요구를 분석해 직접 대답하거나, 주어진 함수를 이용해 필요한 정보를 먼저 확보한 '
             '후 대답하세요.',
  'role': 'system'},
 {'content': '오늘 서울 날씨는 어때?', 'role': 'user'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_NN0x7wRIJWLpB3iUT4aMf8n8', function=Function(arguments='{"city_name":"Seoul","units":"metric"}', name='get_current_weather'), type='function')]),
 {'content': '{"city": "Seoul", "description": "clear sky", "temperature": '
             '-4.24, "temperature_feels_like": -4.24, "humidity": 68}',
  'name': 'get_current_weather',
  'role': 'tool',
  'tool_call_id': 'call_NN0x7wRIJWLpB3iUT4aMf8n8'}]


'오늘 서울의 날씨는 맑은 하늘이며, 기온은 약 -4.2도입니다. 체감 온도도 비슷하게 -4.2도이고, 습도는 68%입니다. 외출 시 따뜻하게 입으시는 것이 좋겠습니다.'